In [ ]:
import pandas as pd
import auriga_public as ap
import numpy as np
import matplotlib.pyplot as plt
import cmasher as cmr
from scipy.stats import binned_statistic_2d

### Example 1: living the stream

In [ ]:
halo = 18
level = 4
snap = 127

outputdir = f'/Users/ahriley/data/auriga/level4/halo_{halo}/'
list_directory = '/Users/ahriley/data/auriga/level4/accretedstardata/'

In [ ]:
# load Riley+25,Shipp+25 stream catalogue
fname = '../data/riley2025-shipp2025-combined-tables.txt'
stream_cat = pd.read_csv(fname, skiprows=16, header=None, sep=' ', names=['halo', 'level', 'id', 'morph', 'mstar', 'npart', 'f_bound', 'peri', 'apo', 't_acc', 'dist', 'flag_sat_sat', 'match_up', 'match_up_flag'])

# get streams for this halo and level
morph = 'stream'
sel = (stream_cat['halo'] == halo) & (stream_cat['level'] == level) & (stream_cat['morph'] == morph)
stream_cat = stream_cat[sel]

stream_cat

In [ ]:
stream_list = stream_cat['id'].to_list()

In [ ]:
# load snapshot
attrstoload = ['Coordinates', 'ParticleIDs']
partType = 4
snapobj = ap.snapshot.load_snapshot(snap, partType, loadlist=attrstoload, snappath=outputdir, verbose=False)

In [ ]:
# load subhalo catalogue, center on host halo
subobj = ap.subhalos.subfind(snap, directory=outputdir, loadlist=['SubhaloPos', 'Group_R_Crit200'])
snapobj = ap.util.CentreOnHalo(snapobj, subobj.data['SubhaloPos'][0])

In [ ]:
# load accreted particle lists
mdata = ap.util.read_starparticle_mergertree_data_hdf5(snap, list_directory, f'halo_{halo}')

In [ ]:
# get particles that are in streams
sel = np.isin(mdata['Exsitu']['PeakMassIndex'], stream_list)
stream_pids = mdata['Exsitu']['ParticleIDs'][sel]

# now get the indices of those particles in the snapshot
sel = np.isin(snapobj.data['ParticleIDs'], stream_pids)

fig, axes = plt.subplots(figsize=(15, 5), ncols=3)

bins = np.linspace(-400, 400, 200)
kwargs = {'bins': bins, 'norm': 'log', 'cmap': cmr.neutral_r}
axes[0].hist2d(snapobj.data['Coordinates'][sel, 0] * 1000, snapobj.data['Coordinates'][sel, 1] * 1000, **kwargs)
axes[1].hist2d(snapobj.data['Coordinates'][sel, 0] * 1000, snapobj.data['Coordinates'][sel, 2] * 1000, **kwargs)
axes[2].hist2d(snapobj.data['Coordinates'][sel, 1] * 1000, snapobj.data['Coordinates'][sel, 2] * 1000, **kwargs)

for ax in axes:
    ax.set_aspect('equal')

In [ ]:
fig, axes = plt.subplots(figsize=(15, 5), ncols=3)

kwargs = {'s': 1, 'alpha': 0.5}

# takes ~10 seconds
for pid in stream_list:
    # most of the time is here
    sel = np.isin(mdata['Exsitu']['PeakMassIndex'], pid)
    stream_pids = mdata['Exsitu']['ParticleIDs'][sel]

    sel = np.isin(snapobj.data['ParticleIDs'], stream_pids)
    axes[0].scatter(snapobj.data['Coordinates'][sel, 0] * 1000, snapobj.data['Coordinates'][sel, 1] * 1000, **kwargs)
    axes[1].scatter(snapobj.data['Coordinates'][sel, 0] * 1000, snapobj.data['Coordinates'][sel, 2] * 1000, **kwargs)
    axes[2].scatter(snapobj.data['Coordinates'][sel, 1] * 1000, snapobj.data['Coordinates'][sel, 2] * 1000, **kwargs)

for ax in axes:
    ax.set_xlim(-400, 400)
    ax.set_ylim(-400, 400)
    ax.set_aspect('equal')

### Example 2: bar hopping

Au-18 has a good bar (Fragkoudi+2025). Let's go find it

In [ ]:
halo = 18
level = 4
snap = 127

outputdir = f'/Users/ahriley/data/auriga/level4/halo_{halo}/'

In [ ]:
# load snapshot
attrstoload = ['Coordinates', 'ParticleIDs', 'GFM_StellarFormationTime', 'Velocities', 'Masses']
partType = 4
snapobj = ap.snapshot.load_snapshot(snap, partType, loadlist=attrstoload, snappath=outputdir, verbose=False)

In [ ]:
# load subhalo catalogue, center on host halo
subobj = ap.subhalos.subfind(snap, directory=outputdir, loadlist=['SubhaloPos', 'Group_R_Crit200'])
snapobj = ap.util.CentreOnHalo(snapobj, subobj.data['SubhaloPos'][0])

In [ ]:
# reminder - we live in a box!
plt.hist2d(snapobj.data['Coordinates'][:, 0], snapobj.data['Coordinates'][:, 1], bins=150, norm='log');

In [ ]:
# only grab things within R200 of the host halo, and only grab stars
snapobj = ap.util.apply_mask(snapobj, stars=True, radialcut=subobj.data['Group_R_Crit200'][0])

# rotate coords to align with the galaxy
xdir, ydir, zdir = ap.util.align_galaxy( snapobj, idx=None, radialcut=0.1*subobj.data['Group_R_Crit200'][0] )
ap.util.rotateto(snapobj, xdir, dir2=ydir, dir3=zdir)

In [ ]:
bins = np.linspace(-10, 10, 200)

fig, axes = plt.subplots(figsize=(15, 5), ncols=3)

axes[0].hist2d(snapobj.data['Coordinates'][:, 1] * 1000, snapobj.data['Coordinates'][:, 2] * 1000, bins=bins, norm='log')
axes[1].hist2d(snapobj.data['Coordinates'][:, 1] * 1000, snapobj.data['Coordinates'][:, 0] * 1000, bins=bins, norm='log')
axes[2].hist2d(snapobj.data['Coordinates'][:, 2] * 1000, snapobj.data['Coordinates'][:, 0] * 1000, bins=bins, norm='log')

for ax in axes:
    ax.set_aspect('equal');

Note that we didn't specifically try to align the y-axis with the bar, but this worked out

### Example 3: do we have chemistry?

In [ ]:
halo = 6
level = 4
snap = 127

outputdir = f'/Users/ahriley/data/auriga/level4/halo_{halo}/'

In [ ]:
# load snapshot
attrstoload = ['Coordinates', 'ParticleIDs', 'GFM_StellarFormationTime', 'Velocities', 'Masses', 'GFM_Metals']
partType = 4
snapobj = ap.snapshot.load_snapshot(snap, partType, loadlist=attrstoload, snappath=outputdir, verbose=False)

In [ ]:
# load subhalo catalogue, center on host halo
subobj = ap.subhalos.subfind(snap, directory=outputdir, loadlist=['SubhaloPos', 'Group_R_Crit200'])
snapobj = ap.util.CentreOnHalo(snapobj, subobj.data['SubhaloPos'][0])

In [ ]:
# only grab things within R200 of the host halo, and only grab stars
snapobj = ap.util.apply_mask(snapobj, stars=True, radialcut=subobj.data['Group_R_Crit200'][0])

# rotate coords to align with the galaxy
xdir, ydir, zdir = ap.util.align_galaxy( snapobj, idx=None, radialcut=0.1*subobj.data['Group_R_Crit200'][0] )
ap.util.rotateto(snapobj, xdir, dir2=ydir, dir3=zdir)

In [ ]:
# GFM_Metals is 9-element array with the following elements: H, He, C, N, O, Ne, Mg, Si, Fe (in this order)
fe = snapobj.data['GFM_Metals'][:,8]
mg = snapobj.data['GFM_Metals'][:,6]
h = snapobj.data['GFM_Metals'][:,0]

# Asplund+2021: https://ui.adsabs.harvard.edu/abs/2021A%26A...653A.141A/abstract
feh = np.log10(fe/h) - np.log10(55.845/1.008) - (7.46-12)  # solar Fe/H = 0.00134/0.7381
mgfe = np.log10(mg/fe) - np.log10(24.305/55.845) - (7.55-7.46)

In [ ]:
# FeH across the host galaxy
bins = np.linspace(-20, 20, 200)
vals = feh

fig, axes = plt.subplots(figsize=(15, 5), ncols=3)

sel = np.isfinite(vals) & ~np.isnan(vals)
bin_kwargs = {'bins': bins, 'statistic': 'median', 'values': vals[sel]}
plot_kwargs = {'origin': 'lower', 'aspect': 'equal', 'cmap': cmr.bubblegum, 'vmin': -2, 'vmax': 1, 'extent': [bins[0], bins[-1], bins[0], bins[-1]]}

stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 1] * 1000, y=snapobj.data['Coordinates'][sel, 2] * 1000, **bin_kwargs)
im = axes[0].imshow(stat.T, **plot_kwargs)
stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 1] * 1000, y=snapobj.data['Coordinates'][sel, 0] * 1000, **bin_kwargs)
axes[1].imshow(stat.T, **plot_kwargs)
stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 2] * 1000, y=snapobj.data['Coordinates'][sel, 0] * 1000, **bin_kwargs)
axes[2].imshow(stat.T, **plot_kwargs)

plt.colorbar(im, ax=axes, label='[Fe/H]')

In [ ]:
# alpha-enhanced halo relative to host galaxy
bins = np.linspace(-200, 200, 200)
vals = mgfe

fig, axes = plt.subplots(figsize=(15, 5), ncols=3)

sel = np.isfinite(vals) & ~np.isnan(vals)
bin_kwargs = {'bins': bins, 'statistic': 'median', 'values': vals[sel]}
plot_kwargs = {'origin': 'lower', 'aspect': 'equal', 'cmap': cmr.bubblegum, 'vmin': -0.4, 'vmax': -0.1, 'extent': [bins[0], bins[-1], bins[0], bins[-1]]}

stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 1] * 1000, y=snapobj.data['Coordinates'][sel, 2] * 1000, **bin_kwargs)
im = axes[0].imshow(stat.T, **plot_kwargs)
stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 1] * 1000, y=snapobj.data['Coordinates'][sel, 0] * 1000, **bin_kwargs)
axes[1].imshow(stat.T, **plot_kwargs)
stat, x_edges, y_edges, binnumber = binned_statistic_2d(x=snapobj.data['Coordinates'][sel, 2] * 1000, y=snapobj.data['Coordinates'][sel, 0] * 1000, **bin_kwargs)
axes[2].imshow(stat.T, **plot_kwargs)

plt.colorbar(im, ax=axes, label='[Mg/Fe]')

### Suggestions

Here are some things that you could hack together pretty quickly:
* Get the stellar mass of all satellites, plot a cumulative stellar mass function ("luminosity" function)
* Plot the age-metallicity relation, maybe different for different objects (satellites, accreted halo, in-situ stars)
* 2D histograms of mass, have different panels correspond to different ages/metallicity cuts

In [ ]:
# go nuts!